In [1]:
import os
import pandas as pd

In [2]:
import datasets

In [6]:
root = "/ptmp/rfechner/out/exp13_rollouts"
dirs = sorted(
    [os.path.join(root, d, 'val_jsonl') for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))]
)


# subsample dirs to just include the grpos
dirs = [d for d in dirs if d.endswith('grpo/val_jsonl')]
dirs

['/ptmp/rfechner/out/exp13_rollouts/eurollm_9b_instruct__grpo/val_jsonl',
 '/ptmp/rfechner/out/exp13_rollouts/llama_3.1_8b_instruct__grpo/val_jsonl',
 '/ptmp/rfechner/out/exp13_rollouts/qwen2.5_1.5b__grpo/val_jsonl']

In [7]:
import pathlib
target_root='/ptmp/rfechner/out/exp13_rollouts_subsampled'
os.makedirs(target_root, exist_ok=True)
def subsample_and_to_disk(path : str) -> None:
    p = pathlib.Path(path)
    filenames = os.listdir(path) # 0_rollouts.jsonl, 10_rollouts.jsonl etc
    for filename in filenames:
        with open(os.path.join(path, filename), 'r') as file:
            df = pd.read_json(file, lines=True)
            subdf = ( # sample 16 rows per question
                df
                .groupby('input', sort=False, group_keys=False)
                .sample(n=16, replace=False, random_state=0)
            )
        outpath = os.path.join(target_root, *p.parts[-2:])
        os.makedirs(outpath, exist_ok=True)
        with open(os.path.join(outpath, filename), 'w') as file:
            subdf.to_json(file, orient='records', lines=True)

In [8]:
from tqdm import tqdm
for d in tqdm(dirs):
    subsample_and_to_disk(d)


100%|██████████| 3/3 [02:10<00:00, 43.54s/it]


In [10]:
qwen_root = '/ptmp/rfechner/out/exp05_rollouts_qwen2.5-7b/'
dirs = [os.path.join(qwen_root, d, 'val_jsonl') for d in [f'qwen2.5_7b__{m}' for m in ['grpo-passk', 'gspo', 'kl-cov', 'grpo-s', 'grpo']]]

# subsample dirs to just include grpo
dirs = [d for d in dirs if d.endswith('grpo/val_jsonl')]
dirs

['/ptmp/rfechner/out/exp05_rollouts_qwen2.5-7b/qwen2.5_7b__grpo/val_jsonl']

In [11]:
def subsample_and_to_disk_qwen(path : str) -> None:
    p = pathlib.Path(path)
    filenames = os.listdir(path) # 0_rollouts.jsonl, 10_rollouts.jsonl etc
    for filename in filenames:
        with open(os.path.join(path, filename), 'r') as file:
            df = pd.read_json(file, lines=True)
            subdf = (
                df
                .groupby('input', sort=False, group_keys=False)
                .sample(n=16, replace=False, random_state=0)
            )
        
        outpath = os.path.join(target_root, *p.parts[-2:])
        os.makedirs(outpath, exist_ok=True)
        with open(os.path.join(outpath, filename), 'w') as file:
            subdf.to_json(file, orient='records', lines=True)

In [12]:
for d in tqdm(dirs):
    subsample_and_to_disk(d)

100%|██████████| 1/1 [00:34<00:00, 34.32s/it]


In [13]:
# also: I'd like to compare likelihoods of rollouts of final models under different final models.
# This should give me a matrix M x K x K where K is the number of methods and M is the number of base models.
# I should just read and concatenate all 'last' rollout files from the exp13_rollouts_subsampled directly, and
# annotate them with the respective source model and source algorithm.
root = '/ptmp/rfechner/out/exp13_rollouts_subsampled'
dirs = sorted(
    [os.path.join(root, d, 'val_jsonl') for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))]
)
dirs

['/ptmp/rfechner/out/exp13_rollouts_subsampled/eurollm_9b_instruct__grpo-passk/val_jsonl',
 '/ptmp/rfechner/out/exp13_rollouts_subsampled/eurollm_9b_instruct__grpo-s/val_jsonl',
 '/ptmp/rfechner/out/exp13_rollouts_subsampled/eurollm_9b_instruct__grpo/val_jsonl',
 '/ptmp/rfechner/out/exp13_rollouts_subsampled/eurollm_9b_instruct__gspo/val_jsonl',
 '/ptmp/rfechner/out/exp13_rollouts_subsampled/eurollm_9b_instruct__kl-cov/val_jsonl',
 '/ptmp/rfechner/out/exp13_rollouts_subsampled/llama_3.1_8b_instruct__grpo-passk/val_jsonl',
 '/ptmp/rfechner/out/exp13_rollouts_subsampled/llama_3.1_8b_instruct__grpo-s/val_jsonl',
 '/ptmp/rfechner/out/exp13_rollouts_subsampled/llama_3.1_8b_instruct__grpo/val_jsonl',
 '/ptmp/rfechner/out/exp13_rollouts_subsampled/llama_3.1_8b_instruct__gspo/val_jsonl',
 '/ptmp/rfechner/out/exp13_rollouts_subsampled/llama_3.1_8b_instruct__kl-cov/val_jsonl',
 '/ptmp/rfechner/out/exp13_rollouts_subsampled/qwen2.5_1.5b__grpo-passk/val_jsonl',
 '/ptmp/rfechner/out/exp13_rollouts_

In [14]:
dfs = []
for d in tqdm(dirs):
    last = sorted(os.listdir(d), key=lambda x: int(x.split('_')[0]))[-1]
    print(d, last)
    with open(os.path.join(d, last), 'r') as file:
        df = pd.read_json(file, lines=True)
    model_method = pathlib.Path(d).parts[-2]
    model, method = model_method.split('__')
    df['model'] = [model] * len(df)
    df['method'] = [method] * len(df)
    dfs.append(df.copy())

  5%|▌         | 1/19 [00:00<00:02,  8.72it/s]

/ptmp/rfechner/out/exp13_rollouts_subsampled/eurollm_9b_instruct__grpo-passk/val_jsonl 80_rollouts.jsonl
/ptmp/rfechner/out/exp13_rollouts_subsampled/eurollm_9b_instruct__grpo-s/val_jsonl 80_rollouts.jsonl


 16%|█▌        | 3/19 [00:00<00:02,  6.87it/s]

/ptmp/rfechner/out/exp13_rollouts_subsampled/eurollm_9b_instruct__grpo/val_jsonl 70_rollouts.jsonl
/ptmp/rfechner/out/exp13_rollouts_subsampled/eurollm_9b_instruct__gspo/val_jsonl 80_rollouts.jsonl


 26%|██▋       | 5/19 [00:00<00:02,  5.92it/s]

...

 89%|████████▉ | 17/19 [00:03<00:00,  5.40it/s]

/ptmp/rfechner/out/exp13_rollouts_subsampled/qwen2.5_7b__grpo/val_jsonl 70_rollouts.jsonl
/ptmp/rfechner/out/exp13_rollouts_subsampled/qwen2.5_7b__gspo/val_jsonl 80_rollouts.jsonl


100%|██████████| 19/19 [00:03<00:00,  5.19it/s]

/ptmp/rfechner/out/exp13_rollouts_subsampled/qwen2.5_7b__kl-cov/val_jsonl 60_rollouts.jsonl


100%|██████████| 19/19 [00:03<00:00,  5.24it/s]


In [15]:
df = pd.concat(dfs)


In [16]:
mapper = lambda row : [{'role' : 'user', 'content' : row['input'].removeprefix('user\n').removesuffix('\nassistant\n')}, {'role' : 'assistant', 'content' : row['output']}]
accu = []
for i, row in df.iterrows():
    accu.append(mapper(row))

In [17]:
df['prompt'] = accu
df = df.drop(['input', 'output', 'extracted_gt', 'format_score'], axis='columns', inplace=False)
df

,gts,score,step,reward,acc,model,method,prompt
0,"\left( 3, \frac{\pi}{2} \right)",0,80,0,0,eurollm_9b_instruct,grpo-passk,"[{'role': 'user', 'content': 'Convert the poin..."
1,"\left( 3, \frac{\pi}{2} \right)",0,80,0,0,eurollm_9b_instruct,grpo-passk,"[{'role': 'user', 'content': 'Convert the poin..."
2,"\left( 3, \frac{\pi}{2} \right)",0,80,0,0,eurollm_9b_instruct,grpo-passk,"[{'role': 'user', 'content': 'Convert the poin..."
3,"\left( 3, \frac{\pi}{2} \right)",1,80,1,1,eurollm_9b_instruct,grpo-passk,"[{'role': 'user', 'content': 'Convert the poin..."
4,"\left( 3, \frac{\pi}{2} \right)",0,80,0,0,eurollm_9b_instruct,grpo-passk,"[{'role': 'user', 'content': 'Convert the poin..."
...,...,...,...,...,...,...,...,...
10075,\sqrt{\frac{95}{24}},0,60,0,0,qwen2.5_7b,kl-cov,"[{'role': 'user', 'content': 'system You are a..."
10076,\sqrt{\frac{95}{24}},0,60,0,0,qwen2.5_7b,kl-cov,"[{'role': 'user', 'content': 'system You are a..."
10077,\sqrt{\frac{95}{24}},0,60,0,0,qwen2.5_7b,kl-cov,"[{'role': 'user', 'content': 'system You are a..."
10078,\sqrt{\frac{95}{24}},0,60,0,0,qwen2.5_7b,kl-cov,"[{'role': 'user', 'content': 'system You are a..."


In [18]:
outpath = '/u/rfechner/data/data_driven_subset'
os.makedirs(outpath, exist_ok=True)


with open(os.path.join(outpath, 'rollouts.jsonl'), 'w') as file:
    df.to_json(file, orient='records', lines=True)

In [19]:
import datasets
ds = datasets.load_dataset('json', data_files=os.path.join(outpath, 'rollouts.jsonl'))['train']

Generating train split: 0 examples [00:00, ? examples/s]